In [1]:
import pandas as pd

path = "../data/static/"
routes = pd.read_csv(path + "routes.txt", dtype=str)
trips = pd.read_csv(path + "trips.txt", dtype=str)
stop_times = pd.read_csv(path + "stop_times.txt", dtype=str)
stops = pd.read_csv(path + "stops.txt", dtype=str)

for name, df in [("routes", routes), ("trips", trips), ("stop_times", stop_times), ("stops", stops)]:
    print(name, df.shape)

routes (230, 9)
trips (129271, 10)
stop_times (4131941, 10)
stops (9128, 12)


In [2]:
display(routes.head())
display(trips.head())
display(stop_times.head())
display(stops.head())

,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_url,route_color,route_text_color
0,1,1,1,LINE 1 SHUTTLE BUS,NaN,3,NaN,FFFFFF,000000
1,10,1,10,Van Horne,NaN,3,NaN,ed1c24,ffffff
2,100,1,100,Flemingdon Park,NaN,3,NaN,ed1c24,ffffff
3,101,1,101,Downsview Park,NaN,3,NaN,ed1c24,ffffff
4,102,1,102,Markham Rd,NaN,3,NaN,ed1c24,ffffff


,trip_id,route_id,service_id,trip_headsign,trip_short_name,direction_id,block_id,shape_id,wheelchair_accessible,bikes_allowed
0,125389020,10,1,East - 10 Van Horne towards Victoria Park,NaN,1,100112,shp-10-03,1,2
1,117101020,10,1,East - 10 Van Horne towards Victoria Park,NaN,1,100112,shp-10-03,1,2
2,12581020,10,1,East - 10 Van Horne towards Victoria Park,NaN,1,100111,shp-10-04,1,2
3,118795020,10,1,East - 10 Van Horne towards Victoria Park,NaN,1,100112,shp-10-03,1,2
4,16945020,10,1,East - 10 Van Horne towards Victoria Park,NaN,1,100112,shp-10-03,1,2


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint
0,1080,25:00:00,25:00:00,3747,1,83 JONES COMMISSIONERS,0,0,0.00,1
1,1080,25:01:10,25:01:10,8865,2,83 JONES COMMISSIONERS,0,0,414.84,0
2,1080,25:02:00,25:02:00,9080,3,83 JONES COMMISSIONERS,0,0,676.86,0
3,1080,25:02:42,25:02:42,5631,4,83 JONES COMMISSIONERS,0,0,895.92,0
4,1080,25:03:39,25:03:39,5904,5,83 JONES COMMISSIONERS,0,0,1191.84,0


,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding
0,2,155,Bathurst St at Horsham Ave,NaN,43.768449,-79.441504,NaN,NaN,NaN,NaN,NaN,1
1,3,1514,Royal York Rd at King Georges Rd,NaN,43.650225,-79.512691,NaN,NaN,NaN,NaN,NaN,1
2,4,2593,Finch Ave West at Driftwood Ave,NaN,43.757924,-79.513523,NaN,NaN,NaN,NaN,NaN,1
3,5,1231,Lawrence Ave East at Wanless Cres (West),NaN,43.727149,-79.392116,NaN,NaN,NaN,NaN,NaN,2
4,7,4090,Kennedy Rd at Huntingwood Dr,NaN,43.795020,-79.293198,NaN,NaN,NaN,NaN,NaN,1


In [3]:
route = routes[routes["route_short_name"] == "29"]
display(route)

route_trips = trips[trips["route_id"].isin(route["route_id"])]
print("班次數量：", len(route_trips))
print(route_trips["service_id"].value_counts())

,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_url,route_color,route_text_color
80,29,1,29,Dufferin,NaN,3,NaN,ed1c24,ffffff


班次數量： 1344
service_id
4    384
2    350
3    319
1    291
Name: count, dtype: int64


In [4]:
one_trip = route_trips["trip_id"].iloc[0]

trip_stops = (
    stop_times[stop_times["trip_id"] == one_trip]
    .merge(stops, on="stop_id")
    .assign(stop_sequence=lambda d: d["stop_sequence"].astype(int))
    .sort_values("stop_sequence")
)
display(trip_stops[["stop_sequence", "arrival_time", "stop_name", "stop_lat", "stop_lon"]])

,stop_sequence,arrival_time,stop_name,stop_lat,stop_lon
0,1,13:25:00,Dufferin Gate Loop,43.634401,-79.425872
1,2,13:26:49,Dufferin St at Liberty St,43.636946,-79.426505
2,3,13:27:55,Dufferin St at King St West,43.638781,-79.427202
3,4,13:28:45,Dufferin St at Melbourne Ave,43.640185,-79.427735
4,5,13:29:51,Dufferin St at Queen St West,43.642008,-79.428427
5,6,13:30:55,Dufferin St at Alma Ave,43.644509,-79.429381
6,7,13:31:47,Dufferin St at Gordon St,43.646696,-79.430224
7,8,13:32:53,Dufferin St at Dundas St West,43.649459,-79.431252
8,9,13:34:53,Dufferin St at College St North Side,43.652873,-79.432579
9,10,13:36:11,Dufferin St at Sylvan Ave North Side,43.655107,-79.433431


In [5]:
import requests
from google.transit import gtfs_realtime_pb2

feed = gtfs_realtime_pb2.FeedMessage()
feed.ParseFromString(requests.get("https://bustime.ttc.ca/gtfsrt/vehicles", timeout=10).content)
print(len(feed.entity), feed.entity[0])

1153 id: "1"
vehicle {
  trip {
    trip_id: "66976070"
    schedule_relationship: SCHEDULED
    route_id: "52"
  }
  position {
    latitude: 43.7058296
    longitude: -79.4906311
    bearing: 255
    speed: 0
  }
  current_stop_sequence: 31
  current_status: IN_TRANSIT_TO
  timestamp: 1792292106
  stop_id: "2235"
  vehicle {
    id: "3639"
  }
  occupancy_status: EMPTY
}



In [6]:
rows = []
for e in feed.entity:
    v = e.vehicle
    rows.append({
        "vehicle_id": v.vehicle.id,
        "trip_id": v.trip.trip_id,
        "route_id": v.trip.route_id,
        "lat": v.position.latitude,
        "lon": v.position.longitude,
        "timestamp": pd.to_datetime(v.timestamp, unit="s", utc=True).tz_convert("America/Toronto"),
    })
rt = pd.DataFrame(rows)
display(rt.head())

matched = rt.merge(trips[["trip_id", "service_id"]], on="trip_id", how="left")
print("對得上時間表嘅車輛比例：", matched["service_id"].notna().mean().round(3))

,vehicle_id,trip_id,route_id,lat,lon,timestamp
0,3639,66976070,52,43.705830,-79.490631,2026-10-17 22:55:06-04:00
1,3631,29021070,165,43.740082,-79.421059,2026-10-17 22:55:05-04:00
2,3634,87463070,927,43.728596,-79.604797,2026-10-17 22:55:01-04:00
3,3636,68588070,84,43.756367,-79.433548,2026-10-17 22:54:59-04:00
4,3637,,,43.745697,-79.532143,2026-10-17 22:54:54-04:00


對得上時間表嘅車輛比例： 0.611


In [7]:
unmatched = matched[matched["service_id"].isna()]

print("冇 trip_id 嘅車：", (rt["trip_id"] == "").sum())
print("有 trip_id 但對唔上：", (unmatched["trip_id"] != "").sum())

display(unmatched[unmatched["trip_id"] != ""][["trip_id", "route_id"]].head(10))
print("靜態表 trip_id 例子：", trips["trip_id"].head(5).tolist())

cal = pd.read_csv(path + "calendar.txt", dtype=str)
print("時間表有效期：", cal["start_date"].min(), "至", cal["end_date"].max())

冇 trip_id 嘅車： 448
有 trip_id 但對唔上： 0


,trip_id,route_id


靜態表 trip_id 例子： ['125389020', '117101020', '12581020', '118795020', '16945020']
時間表有效期： 20260930 至 20261031


In [9]:
print("而家喺路上：", (rt["route_id"] == route["route_id"].iloc[0]).sum(), "架")


而家喺路上： 14 架
